# InsertionSort (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch12_running_time/InsertionSort_cpp.ipynb)

Chapter 12 — Running time. Runnable notebook of `InsertionSort`: the algorithm in C++, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/analysis/12-running-time/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch12_running_time/InsertionSort_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch12_running_time/InsertionSort_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch12_running_time_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code the arrays are indexed from 1; in the C++ code from 0: the element `a[i]` of the pseudo-code is `a[i-1]` in the code (and a loop `while i <= n` from 1 becomes `while i < n` from 0). The printed tables use the indices of the pseudo-code.

## Insertion sort: InsertionSort

**Input:** an array $a[1..n]$ of $n$ natural numbers. **Output:** the array $a$ sorted in non-decreasing order. If $t(i)$ is the number of shifts at iteration $i$, then $S(n) = 6n - 4 + 3\sum_{i=2}^{n} t(i)$; in the worst case (decreasing array) $S(n) = \frac{3n^2+9n-8}{2} = \Theta(n^2)$.

Pseudo-code: section 12.3 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/12-running-time/).

In [1]:
%%writefile insertion_sort.cpp
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Line | Executions\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Total steps S = %lld\n", Total());
}

// "[a[from], ..., a[to-1]]"
string Fmt(const int* a, int from, int to)
{
    string s = "[";
    for (int i = from; i < to; i++)
    {
        if (i > from) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// InsertionSort(a, n)
void InsertionSort(int a[], int n, bool trace = false)
{
    long long prev = 0;                    // only for the trace
    int i = 1; cnt[2]++;                   // i := 2 in the pseudo-code
    cnt[3]++;
    while (i < n)                          // i <= n in the pseudo-code
    {
        int key = a[i]; cnt[4]++;
        int j = i - 1; cnt[5]++;
        cnt[6]++;                          // the while condition is counted at every evaluation
        while (j >= 0 && a[j] > key)       // j >= 1 and a[j] > key in the pseudo-code
        {
            a[j + 1] = a[j]; cnt[7]++;
            j = j - 1; cnt[8]++;
            cnt[6]++;
        }
        a[j + 1] = key; cnt[10]++;
        i = i + 1; cnt[11]++;
        cnt[3]++;
        if (trace)                         // trace: after the increment, i is the pseudo-code index of the iteration just finished
        {
            printf("%2d | %3d | %4lld | %s\n", i, key, cnt[7] - prev, Fmt(a, 0, n).c_str());
            prev = cnt[7];
        }
    }
}

long long StepsOf(const int a[], int n)    // works on a copy of a
{
    int b[64];
    for (int i = 0; i < n; i++) b[i] = a[i];
    Reset();
    InsertionSort(b, n);
    return Total();
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Example of the lecture notes: a[1..6] = (5, 2, 4, 6, 1, 3)
    int a[] = {5, 2, 4, 6, 1, 3};
    int n = 6;
    printf("a = %s\n", Fmt(a, 0, n).c_str());
    Reset();
    printf(" i | key | t(i) | array after the iteration\n");
    InsertionSort(a, n, true);
    printf("sorted: %s\n", Fmt(a, 0, n).c_str());
    PrintSteps({2, 3, 4, 5, 6, 7, 8, 10, 11});
    long long t = cnt[7];
    printf("sum t(i) = %lld, S(n) = 6n - 4 + 3 sum t(i) = %lld\n", t, 6 * n - 4 + 3 * t);

    // Worst case: decreasing array, t(i) = i - 1
    printf("\n");
    int d[] = {6, 5, 4, 3, 2, 1};
    printf("a = %s (decreasing): S = %lld = (3n^2 + 9n - 8)/2 = %d\n", Fmt(d, 0, n).c_str(), StepsOf(d, n), (3 * n * n + 9 * n - 8) / 2);
    printf("sum t(i) = %lld = n(n-1)/2 = %d\n", cnt[7], n * (n - 1) / 2);

    printf("\n");
    printf("Decreasing array (n, ..., 2, 1):\n");
    printf("  n | S(n) | (3n^2 + 9n - 8)/2\n");
    for (int k = 1; k <= 8; k++)
    {
        int b[8];
        for (int i = 0; i < k; i++) b[i] = k - i;
        printf("%3d | %4lld | %d\n", k, StepsOf(b, k), (3 * k * k + 9 * k - 8) / 2);
    }
    return 0;
}
#endif

Writing insertion_sort.cpp


In [2]:
!g++ -std=c++17 -O2 -o insertion_sort insertion_sort.cpp && ./insertion_sort

a = [5, 2, 4, 6, 1, 3]
 i | key | t(i) | array after the iteration
 2 |   2 |    1 | [2, 5, 4, 6, 1, 3]
 3 |   4 |    1 | [2, 4, 5, 6, 1, 3]
 4 |   6 |    0 | [2, 4, 5, 6, 1, 3]
 5 |   1 |    4 | [1, 2, 4, 5, 6, 3]
 6 |   3 |    3 | [1, 2, 3, 4, 5, 6]
sorted: [1, 2, 3, 4, 5, 6]
Line | Executions
   2 |          1
   3 |          6
   4 |          5
   5 |          5
   6 |         14
   7 |          9
   8 |          9
  10 |          5
  11 |          5
Total steps S = 59
sum t(i) = 9, S(n) = 6n - 4 + 3 sum t(i) = 59

a = [6, 5, 4, 3, 2, 1] (decreasing): S = 77 = (3n^2 + 9n - 8)/2 = 77
sum t(i) = 15 = n(n-1)/2 = 15

Decreasing array (n, ..., 2, 1):
  n | S(n) | (3n^2 + 9n - 8)/2
  1 |    2 | 2
  2 |   11 | 11
  3 |   23 | 23
  4 |   38 | 38
  5 |   56 | 56
  6 |   77 | 77
  7 |  101 | 101
  8 |  128 | 128


**Try it yourself.** Change the array and check the formula $6n - 4 + 3\sum t(i)$.

In [3]:
%%writefile prova_insertion_sort.cpp
#define SOLO_LIBRERIA   // do not run the main of insertion_sort.cpp
#include "insertion_sort.cpp"

int main()
{
    int a[] = {8, 3, 5, 1, 9, 2, 7};      // <-- change the array
    int n = 7;
    Reset();
    printf(" i | key | t(i) | array after the iteration\n");
    InsertionSort(a, n, true);
    PrintSteps({2, 3, 4, 5, 6, 7, 8, 10, 11});
    printf("6n - 4 + 3 sum t(i) = %lld\n", 6 * n - 4 + 3 * cnt[7]);
    return 0;
}

Writing prova_insertion_sort.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_insertion_sort prova_insertion_sort.cpp && ./prova_insertion_sort

 i | key | t(i) | array after the iteration
 2 |   3 |    1 | [3, 8, 5, 1, 9, 2, 7]
 3 |   5 |    1 | [3, 5, 8, 1, 9, 2, 7]
 4 |   1 |    3 | [1, 3, 5, 8, 9, 2, 7]
 5 |   9 |    0 | [1, 3, 5, 8, 9, 2, 7]
 6 |   2 |    4 | [1, 2, 3, 5, 8, 9, 7]
 7 |   7 |    2 | [1, 2, 3, 5, 7, 8, 9]
Line | Executions
   2 |          1
   3 |          7
   4 |          6
   5 |          6
   6 |         17
   7 |         11
   8 |         11
  10 |          6
  11 |          6
Total steps S = 71
6n - 4 + 3 sum t(i) = 71


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).